---
## Python Packages & Directories
---

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

from __future__ import (division, print_function, absolute_import,
                        unicode_literals)
import  sys, os, glob, time, IPython

import astropy.constants as const
import astropy.units as u
from astropy.io import fits
from astropy.io import ascii
from astropy.table import Table
# from astropy.utils.data import get_pkg_data_filename
from astropy.coordinates import SkyCoord, EarthLocation

# from PyAstronomy import pyasl

import matplotlib as mpl
import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec

import numpy as np
import scipy as sp
import pandas as pd

import seaborn as sns
sns.set_palette("colorblind")
colors = sns.color_palette("colorblind", 20)

# from smh import Session

from spag.read_data import *
from spag.convert import *
from spag.utils import *
from spag.calculate import *
import spag.read_data as rd
import spag.coordinates as scoord

# import alexmods.read_data as rd

script_dir = "/".join(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"].split("/")[:-1]) + "/"
# script_dir = os.path.dirname(os.path.realpath(__file__))+"/"
data_dir = '/Users/ayelland/Research/metal-poor-stars/spag/data/'

with open('/Users/ayelland/Research/metal-poor-stars/project/carbon-project-2025/create-tables-0-date.txt', 'r') as f:
    date = f.readline().strip()


In [2]:
## Show all columns and rows of the dataframe
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows

---
## Load the CSV Datafiles
---

In [16]:
query_basepath = "/Users/ayelland/Research/metal-poor-stars/spag/data/abundances/shetrone2003/"
query_df = pd.read_csv(f"{query_basepath}/astroquery.csv", comment="#")

---
## Preparing the Columns
---

In [4]:
target_stars = query_df.copy()
print(f"Total number of unique target stars: {len(target_stars)}")

Total number of unique target stars: 19


In [5]:
# ## Filter and Re-order the columns for the final target list
# auxcols = [
#     # 'Name', 
#     'Simbad_Identifier', 
#     'RA_hms', 
#     'DEC_dms', 
#     'RA_deg',
#     'DEC_deg',
#     'Loc', 
#     'System', 
#     'Teff',
#     'logg', 
#     'Vmic', 
#     'Fe/H', 
#     'M/H', 
#     'Class',
#     'f_ulc',
#     'f_ulfe',
#     'f_dtrans',
#     'Reference', 
# ]
# datacols = [
#     '[Fe/H]',
#     'ul[Fe/H]', 
#     '[C/H]',
#     'ul[C/H]', 
#     '[C/H]f', 
#     'ul[C/H]f', 
#     '[C/Fe]', 
#     'll[C/Fe]', 
#     'ul[C/Fe]',
#     '[C/Fe]f', 
#     'll[C/Fe]f', 
#     'ul[C/Fe]f', 
#     '[Ba/H]', 
#     'ul[Ba/H]', 
#     '[Ba/Fe]', 
#     'll[Ba/Fe]', 
#     'ul[Ba/Fe]', 
#     '[Sr/H]', 
#     'ul[Sr/H]', 
#     '[Sr/Fe]', 
#     'll[Sr/Fe]', 
#     'ul[Sr/Fe]',
#     '[Eu/H]', 
#     'ul[Eu/H]', 
#     '[Eu/Fe]', 
#     'll[Eu/Fe]',
#     'ul[Eu/Fe]', 
#     'Dtrans_l', 
#     'Dtrans_llim', 
#     'Dtrans_u', 
#     'Dtrans_ulim'
# ]
# target_stars = target_stars[auxcols + datacols].copy()


In [6]:
# ## Data columns
# data_cols = [col for col in list(target_stars.columns)[len(auxcols):] if col != 'epsc_c']

# ## Group columns intelligently
# grouped_cols = []
# i = 0
# while i < len(data_cols):
#     current_col = data_cols[i]
#     group = [current_col]
    
#     # Special handling for Dtrans columns
#     if current_col.startswith('Dtrans_'): # handling for Dtrans columns
        
#         if current_col.endswith('_l'):
#             # Look for corresponding _llim
#             base_name = current_col[:-2]  # Remove '_l'
#             llim_col = base_name + '_llim'
#             if i + 1 < len(data_cols) and data_cols[i + 1] == llim_col:
#                 group.append(data_cols[i + 1])
#                 i += 1
#         elif current_col.endswith('_u'):
#             # Look for corresponding _ulim  
#             base_name = current_col[:-2]  # Remove '_u'
#             ulim_col = base_name + '_ulim'
#             if i + 1 < len(data_cols) and data_cols[i + 1] == ulim_col:
#                 group.append(data_cols[i + 1])
#                 i += 1
#     else: # handling for ll/ul prefixed columns
    
#         # Check for lower limit column (ll prefix)
#         if i + 1 < len(data_cols) and data_cols[i + 1].startswith('ll'):
#             group.append(data_cols[i + 1])
#             i += 1
#         # Check for upper limit column (ul prefix)
#         if i + 1 < len(data_cols) and data_cols[i + 1].startswith('ul'):
#             group.append(data_cols[i + 1])
#             i += 1
    
#     grouped_cols.append(group)
#     i += 1

# display(grouped_cols)

# ## Merge the columns appropriately
# for group in grouped_cols:
#     base_col = group[0]
    
#     if len(group) == 2:  # Pair: base + ul
#         ul_col = group[1]
#         target_stars[base_col] = target_stars[base_col].fillna(
#             target_stars[ul_col].where(target_stars[ul_col].notna()).apply(lambda x: f'<{x}' if pd.notna(x) else np.nan)
#         )
#         target_stars.drop(columns=[ul_col], inplace=True)
        
#     elif len(group) == 3:  # Triplet: base + ll + ul
#         ll_col, ul_col = group[1], group[2]
#         target_stars[base_col] = target_stars[base_col].fillna(
#             target_stars[ll_col].where(target_stars[ll_col].notna()).apply(lambda x: f'>{x}' if pd.notna(x) else np.nan)
#         ).fillna(
#             target_stars[ul_col].where(target_stars[ul_col].notna()).apply(lambda x: f'<{x}' if pd.notna(x) else np.nan)
#         )
#         target_stars.drop(columns=[ll_col, ul_col], inplace=True)

# ## Merge the 'Fe/H' and 'M/H' columns into 'Fe/H'
# if 'M/H' in target_stars.columns and 'Fe/H' in target_stars.columns:
#     target_stars['M/H'] = target_stars['M/H'].fillna(target_stars['Fe/H'])
#     target_stars.drop(columns=['Fe/H'], inplace=True)

# # display(target_stars)

---
## Get the Magnitude Data by Querying Databases (Simbad, Vizier, 2MASS, Gaia, etc.)
---

In [7]:
query_by = 'identifier'
# query_by = 'coordinates'

# identify_by = 'Name'
identify_by = 'Simbad_Identifier'


### Query SIMBAD

In [8]:
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u

## Setup Simbad fields
Simbad.ROW_LIMIT = 10000
Simbad.add_votable_fields(
    'U', 'B', 'V', 'R', 'I', 'J', 'H', 'K',
    'otype', 'sp', 'ra', 'dec', 'pmra', 'pmdec', 'plx_value', 'rvz_nature', 'rvz_qual'
)

## Iterate over target stars
results_list = []
for idx, row in target_stars.iterrows():

    if query_by == 'identifier':
        identifier = row.get(identify_by, f'coord_{idx}').strip()
        name = row['Name'] if 'Name' in row else None
        if 'RA_hms' in row and 'DEC_dms' in row:
            ra_hms = row['RA_hms']
            dec_dms = row['DEC_dms']
        try:
            result = Simbad.query_object(identifier)
            if (result is not None) and len(result) > 0:
                df = result.to_pandas()
                df['Found'] = True
                df['Name'] = name
                df['Query_ID'] = identifier
                df['RA_input'] = ra_hms if 'ra_hms' in locals() else None
                df['DEC_input'] = dec_dms if 'dec_dms' in locals() else None
            else:
                df = pd.DataFrame([{
                    'Found': False,
                    'Name': name,
                    'Query_ID': identifier, 
                    'RA_input': ra_hms if 'ra_hms' in locals() else None,
                    'DEC_input': dec_dms if 'dec_dms' in locals() else None 
                }])
        except Exception as e:
            df = pd.DataFrame([{
                'Found': False, 
                'Name': name,
                'Query_ID': identifier,
                'RA_input': ra_hms if 'ra_hms' in locals() else None,
                'DEC_input': dec_dms if 'dec_dms' in locals() else None,
                'Error': str(e)
            }])
        results_list.append(df)

    elif query_by == 'coordinates':
        identifier = row.get(identify_by, f'coord_{idx}').strip()
        ra_hms = row['RA_hms']
        dec_dms = row['DEC_dms']
        try:
            coord = SkyCoord(ra=ra_hms, dec=dec_dms, unit=(u.hourangle, u.deg))
            result = Simbad.query_region(coord, radius='5s')
            if result is not None:
                df = result.to_pandas()
                df['Found'] = True
                df['Query_ID'] = identifier
                df['RA_input'] = ra_hms
                df['DEC_input'] = dec_dms
            else:
                df = pd.DataFrame([{
                    'Found': False,
                    'Query_ID': identifier,
                    'RA_input': ra_hms,
                    'DEC_input': dec_dms
                }])
        except Exception as e:
            df = pd.DataFrame([{
                'Found': False,
                'Query_ID': identifier,
                'RA_input': ra_hms,
                'DEC_input': dec_dms,
                'Error': str(e)
            }])
        results_list.append(df)

# Combine results and reorder columns
simbad_df = pd.concat(results_list, ignore_index=True)

priority_cols = ['Found', 'Name', 'Query_ID', 'RA_input', 'DEC_input']
for col in priority_cols:
    if col not in simbad_df.columns:
        simbad_df[col] = pd.NA
cols = simbad_df.columns.tolist()

for col in reversed(priority_cols):
    if col in cols:
        cols.insert(0, cols.pop(cols.index(col)))
simbad_df = simbad_df[cols]

Found.
Found.
Found.
Found.
Found.
Found.
Found.
Found.
Found.
Found.
Found.


Found.
Found.
Found.
Found.
Found.
Found.
Found.


In [17]:
display(simbad_df)
# display(list(simbad_df.columns))
simbad_df.to_csv(f"{query_basepath}/simbad_query_results.csv", index=False)

,Found,Name,Query_ID,RA_input,DEC_input,main_id,ra,dec,coo_err_maj,coo_err_min,coo_err_angle,coo_wavelength,coo_bibcode,J,V,I,R,K,H,U,B,pmra,otype,rvz_nature,plx_value,rvz_qual,pmdec,sp_qual,sp_type,sp_bibcode,matched_id
0,True,Car-10,[MOP93] 10,06:41:46.3699,-51:01:22.684,[MOP93] 10,100.443208,-51.022968,0.0598,0.0613,90.0,O,2020yCat.1350....0G,NaN,17.906000,16.497000,NaN,14.967,NaN,NaN,19.152000,0.396,RG*,s,NaN,A,0.168,,,,[MOP93] 10
1,True,Car-12,[MOP93] 12,06:41:36.4795,-50:56:23.180,[MOP93] 12,100.401998,-50.939772,0.0658,0.0599,90.0,O,2020yCat.1350....0G,15.596000,17.907000,16.545000,NaN,14.772,14.844000,NaN,19.174000,0.605,RG*,s,0.0487,A,0.121,,,,[MOP93] 12
2,True,Car-2,[SMS86] 62,06:41:57.8030,-50:59:53.157,[SMS86] 62,100.490846,-50.998099,0.0565,0.0552,90.0,O,2020yCat.1350....0G,15.151000,17.684999,16.222000,NaN,14.424,14.571000,NaN,19.004999,0.563,RG*,sa,0.0496,A,0.130,,,,[SMS86] 62
3,True,Car-3,[MOP93] 3,06:41:54.5940,-50:57:00.688,[MOP93] 3,100.477475,-50.950191,0.0570,0.0518,90.0,O,2020yCat.1350....0G,15.091000,17.684999,16.098000,NaN,14.275,14.488000,NaN,19.146999,0.477,RG*,sa,NaN,A,0.173,,,,[MOP93] 3
4,True,Car-4,[MOP93] 4,06:41:48.2299,-50:55:01.672,[SMS86] 70,100.450958,-50.917131,0.0583,0.0526,90.0,O,2020yCat.1350....0G,15.231000,17.625999,16.181000,NaN,14.512,14.526000,NaN,18.979000,0.352,RR*,sa,NaN,A,0.200,,,,[MOP93] 4
5,True,Fnx-12,2MASS J02401001-3429589,02:40:10.0211,-34:29:58.971,2MASS J02401001-3429589,40.041755,-34.499714,0.0747,0.1026,90.0,O,2020yCat.1350....0G,15.937000,18.200001,16.750000,18.059999,14.977,15.198000,NaN,19.760000,1.312,RG*,s,0.1025,A,-0.978,,,,2MASS J02401001-3429589
6,True,Fnx-21,WEL 112,02:40:04.4134,-34:27:11.601,WEL 112,40.018389,-34.453223,0.0502,0.0716,90.0,O,2020yCat.1350....0G,15.632000,18.370001,NaN,17.414000,14.758,14.833000,NaN,19.945999,0.491,RG*,s,0.0255,A,-0.319,D,K5,1990A&A...233...21L,WEL 112
7,True,Fnx-25,[MOW91] 25,02:39:47.1107,-34:31:49.878,2MASS J02394711-3431499,39.946295,-34.530522,0.0575,0.0782,90.0,O,2020yCat.1350....0G,15.924000,18.590000,NaN,17.746000,15.345,15.204000,NaN,20.268999,0.294,RG*,s,0.1431,A,-0.223,,,,[MOW91] 25
8,True,Leo-2,[MOV98] 2,10:08:33.5218,+12:20:44.801,[MOV98] 2,152.139674,12.345778,0.2399,0.1984,90.0,O,2020yCat.1350....0G,16.766001,19.480000,18.110001,NaN,15.937,16.462000,NaN,NaN,-0.379,*,,NaN,D,0.004,,,,[MOV98] 2
9,True,Leo-5,[MOV98] 5,10:08:22.0511,+12:20:20.815,[MOV98] 5,152.091880,12.339115,0.2228,0.1937,90.0,O,2020yCat.1350....0G,16.591999,19.077999,17.684999,NaN,15.680,16.120001,NaN,20.530001,-0.439,RG*,,NaN,D,-0.075,,,,[MOV98] 5


### Query Gaia DR3

In [11]:
# example: query Gaia DR3 by Gaia source_id string using astroquery.gaia
from astroquery.gaia import Gaia
import pandas as pd
import re

# optional: set a larger timeout if needed
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"   # not required but documents intent
Gaia.ROW_LIMIT = -1

def query_gaia_by_identifier(identifier, columns=None):
    """
    identifier: str, e.g. "Gaia DR3 5480105831331104384" or just the numeric id as str/int
    columns: list or None, ADQL column names to request (None -> request a small useful set)
    returns: pandas.DataFrame (possibly empty) or raises on severe errors
    """
    # try to extract numeric source_id from common forms
    m = re.search(r'(\d{9,22})', str(identifier))  # Gaia DR3 source_id is a long int (>=9 digits)
    if not m:
        raise ValueError(f"Could not find a numeric Gaia source_id in '{identifier}'")

    source_id = int(m.group(1))
    # default columns if none specified
    if columns is None:
        columns = [
            "source_id", 
            "ra", 
            "dec", 
            "parallax", 
            "parallax_error",
            "pmra",
            "pmdec", 
            "radial_velocity",
            "phot_g_mean_mag", 
            "phot_bp_mean_mag", 
            "phot_rp_mean_mag"
        ]
    col_str = ", ".join(columns)

    adql = f"SELECT {col_str} FROM gaiadr3.gaia_source WHERE source_id = {source_id}"

    job = Gaia.launch_job_async(adql)      # launches ADQL job
    tbl = job.get_results()
    if len(tbl) == 0:
        return pd.DataFrame()             # empty -> not found
    df = tbl.to_pandas()
    return df

def query_gaia_by_coordinates(ra_deg, dec_deg, radius_arcsec=2, columns=None):
    """
    ra_deg, dec_deg: float, coordinates in degrees
    radius_arcsec: float, search radius in arcseconds
    columns: list or None, ADQL column names to request (None -> request a small useful set)
    returns: pandas.DataFrame (possibly empty) or raises on severe errors
    """
    if columns is None:
        columns = [
            "source_id", 
            "ra", 
            "dec", 
            "parallax", 
            "parallax_error",
            "pmra",
            "pmdec", 
            "radial_velocity",
            "phot_g_mean_mag", 
            "phot_bp_mean_mag", 
            "phot_rp_mean_mag"
        ]
    col_str = ", ".join(columns)

    radius_deg = radius_arcsec / 3600.0
    adql = f"""
    SELECT {col_str} 
    FROM gaiadr3.gaia_source 
    WHERE 1=CONTAINS(
        POINT('ICRS', ra, dec), 
        CIRCLE('ICRS', {ra_deg}, {dec_deg}, {radius_deg})
    )
    """

    job = Gaia.launch_job_async(adql)      # launches ADQL job
    tbl = job.get_results()
    if len(tbl) == 0:
        return pd.DataFrame()             # empty -> not found
    df = tbl.to_pandas()
    return df

## Iterate over target stars to query Gaia by identifier or coordinates
results_list = []
for idx, row in target_stars.iterrows():
    if query_by == 'identifier':
        identifier = row.get(identify_by, f'coord_{idx}').strip()
        name = row['Name'] if 'Name' in row else None
        try:
            result = query_gaia_by_identifier(identifier)
            if result.empty:
                df = pd.DataFrame([{
                    'Found': False,
                    'Name': name,
                    'Query_ID': identifier
                }])
            else:
                df = result.copy()
                df['Found'] = True
                df['Name'] = name
                df['Query_ID'] = identifier
        except Exception as e:
            df = pd.DataFrame([{
                'Found': False,
                'Name': name,
                'Query_ID': identifier,
                'Error': str(e)
            }])
        results_list.append(df)

    elif query_by == 'coordinates':
        identifier = row.get(identify_by, f'coord_{idx}').strip()
        ra_deg = row['RA_deg']
        dec_deg = row['DEC_deg']
        try:
            result = query_gaia_by_coordinates(ra_deg, dec_deg, radius_arcsec=2)
            if result.empty:
                df = pd.DataFrame([{
                    'Found': False,
                    'Query_ID': identifier,
                    'RA_input': ra_deg,
                    'DEC_input': dec_deg
                }])
            else:
                df = result.copy()
                df['Found'] = True
                df['Query_ID'] = identifier
                df['RA_input'] = ra_deg
                df['DEC_input'] = dec_deg
        except Exception as e:
            df = pd.DataFrame([{
                'Found': False,
                'Query_ID': identifier,
                'RA_input': ra_deg,
                'DEC_input': dec_deg,
                'Error': str(e)
            }])
        results_list.append(df)

# Combine results and reorder columns
gaia_df = pd.concat(results_list, ignore_index=True)

priority_cols = ['Found', 'Name', 'Query_ID', 'RA_input', 'DEC_input']
for col in priority_cols:
    if col not in gaia_df.columns:
        gaia_df[col] = pd.NA
cols = gaia_df.columns.tolist()

for col in reversed(priority_cols):
    if col in cols:
        cols.insert(0, cols.pop(cols.index(col)))
gaia_df = gaia_df[cols]


INFO: Query finished. [astroquery.utils.tap.core]


In [18]:
display(gaia_df)
gaia_df.to_csv(f"{query_basepath}/gaia_query_results.csv", index=False)

,Found,Name,Query_ID,RA_input,DEC_input,Error,source_id,ra,dec,parallax,parallax_error,pmra,pmdec,radial_velocity,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag
0,False,Car-10,[MOP93] 10,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,False,Car-12,[MOP93] 12,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,Car-2,[SMS86] 62,<NA>,<NA>,Could not find a numeric Gaia source_id in '[S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,Car-3,[MOP93] 3,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,Car-4,[MOP93] 4,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,False,Fnx-12,2MASS J02401001-3429589,<NA>,<NA>,Could not find a numeric Gaia source_id in '2M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,False,Fnx-21,WEL 112,<NA>,<NA>,Could not find a numeric Gaia source_id in 'WE...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,False,Fnx-25,[MOW91] 25,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,False,Leo-2,[MOV98] 2,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,False,Leo-5,[MOV98] 5,<NA>,<NA>,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
display(gaia_df.columns)

Index(['Found', 'Name', 'Query_ID', 'RA_input', 'DEC_input', 'Error',
       'source_id', 'ra', 'dec', 'parallax', 'parallax_error', 'pmra', 'pmdec',
       'radial_velocity', 'phot_g_mean_mag', 'phot_bp_mean_mag',
       'phot_rp_mean_mag'],
      dtype='str')

### Merge the Magnitude Data into the Target List

In [14]:
target_stars_out_cols = list(set(simbad_df.columns.tolist() + gaia_df.columns.tolist()))
target_stars_out = pd.DataFrame(columns=target_stars_out_cols)

for i, row in target_stars.iterrows():
    identifier = row.get(identify_by, f'coord_{i}').strip()
    simbad_matches = simbad_df[simbad_df['Query_ID'] == identifier]
    gaia_matches = gaia_df[gaia_df['Query_ID'] == identifier]

    print(i, identifier)
    target_stars_out.at[i, 'Query_ID'] = identifier
    if not simbad_matches.empty:
        for col in simbad_matches.columns:
            # if col in ['FLUX_U','FLUX_B','FLUX_V','FLUX_R','FLUX_I','FLUX_J','FLUX_H','FLUX_K']:
            #     target_stars_out.at[i, col.replace('FLUX_', '')+'mag'] = normal_round(simbad_matches.iloc[0][col], 1)
            target_stars_out.at[i, col] = simbad_matches.iloc[0][col]

    if not gaia_matches.empty:
        for col in gaia_matches.columns:
            # if col in ['phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag']:
            #     target_stars_out.at[i, col.replace('phot_', '').replace('_mean_mag', '').upper()+'mag'] = normal_round(gaia_matches.iloc[0][col], 1)
            target_stars_out.at[i, col] = gaia_matches.iloc[0][col]
            
display(sorted(list(target_stars_out.columns)))
display(target_stars_out)

0 [MOP93] 10
1 [MOP93] 12
2 [SMS86] 62
3 [MOP93] 3
4 [MOP93] 4
5 2MASS J02401001-3429589
6 WEL 112
7 [MOW91] 25
8 [MOV98] 2
9 [MOV98] 5
10 NGC 7099 4
11 Gaia DR3 6751341178000835072
12 2MASS J19401265-3100287
13 Cl* NGC 4590 ALC 53
14 2MASS J01001708-3345139
15 SCMS 1023
16 SCMS 966
17 SCMS 1088
18 SCMS 648


['B',
 'DEC_input',
 'Error',
 'Found',
 'H',
 'I',
 'J',
 'K',
 'Name',
 'Query_ID',
 'R',
 'RA_input',
 'U',
 'V',
 'coo_bibcode',
 'coo_err_angle',
 'coo_err_maj',
 'coo_err_min',
 'coo_wavelength',
 'dec',
 'main_id',
 'matched_id',
 'otype',
 'parallax',
 'parallax_error',
 'phot_bp_mean_mag',
 'phot_g_mean_mag',
 'phot_rp_mean_mag',
 'plx_value',
 'pmdec',
 'pmra',
 'ra',
 'radial_velocity',
 'rvz_nature',
 'rvz_qual',
 'source_id',
 'sp_bibcode',
 'sp_qual',
 'sp_type']

,sp_qual,sp_type,V,DEC_input,plx_value,K,sp_bibcode,phot_g_mean_mag,H,Error,U,Query_ID,B,ra,phot_bp_mean_mag,RA_input,Found,rvz_qual,matched_id,source_id,dec,rvz_nature,pmdec,coo_wavelength,pmra,coo_err_maj,J,main_id,radial_velocity,I,coo_err_angle,Name,parallax_error,phot_rp_mean_mag,parallax,coo_bibcode,otype,R,coo_err_min
0,,,17.906,<NA>,NaN,14.967,,NaN,NaN,Could not find a numeric Gaia source_id in '[M...,NaN,[MOP93] 10,19.152,NaN,NaN,<NA>,False,A,[MOP93] 10,NaN,NaN,s,NaN,O,NaN,0.0598,NaN,[MOP93] 10,NaN,16.497,90.0,Car-10,NaN,NaN,NaN,2020yCat.1350....0G,RG*,NaN,0.0613
1,,,17.907,<NA>,0.0487,14.772,,NaN,14.844,Could not find a numeric Gaia source_id in '[M...,NaN,[MOP93] 12,19.174,NaN,NaN,<NA>,False,A,[MOP93] 12,NaN,NaN,s,NaN,O,NaN,0.0658,15.596,[MOP93] 12,NaN,16.545,90.0,Car-12,NaN,NaN,NaN,2020yCat.1350....0G,RG*,NaN,0.0599
2,,,17.684999,<NA>,0.0496,14.424,,NaN,14.571,Could not find a numeric Gaia source_id in '[S...,NaN,[SMS86] 62,19.004999,NaN,NaN,<NA>,False,A,[SMS86] 62,NaN,NaN,sa,NaN,O,NaN,0.0565,15.151,[SMS86] 62,NaN,16.222,90.0,Car-2,NaN,NaN,NaN,2020yCat.1350....0G,RG*,NaN,0.0552
3,,,17.684999,<NA>,NaN,14.275,,NaN,14.488,Could not find a numeric Gaia source_id in '[M...,NaN,[MOP93] 3,19.146999,NaN,NaN,<NA>,False,A,[MOP93] 3,NaN,NaN,sa,NaN,O,NaN,0.057,15.091,[MOP93] 3,NaN,16.098,90.0,Car-3,NaN,NaN,NaN,2020yCat.1350....0G,RG*,NaN,0.0518
4,,,17.625999,<NA>,NaN,14.512,,NaN,14.526,Could not find a numeric Gaia source_id in '[M...,NaN,[MOP93] 4,18.979,NaN,NaN,<NA>,False,A,[MOP93] 4,NaN,NaN,sa,NaN,O,NaN,0.0583,15.231,[SMS86] 70,NaN,16.181,90.0,Car-4,NaN,NaN,NaN,2020yCat.1350....0G,RR*,NaN,0.0526
5,,,18.200001,<NA>,0.1025,14.977,,NaN,15.198,Could not find a numeric Gaia source_id in '2M...,NaN,2MASS J02401001-3429589,19.76,NaN,NaN,<NA>,False,A,2MASS J02401001-3429589,NaN,NaN,s,NaN,O,NaN,0.0747,15.937,2MASS J02401001-3429589,NaN,16.75,90.0,Fnx-12,NaN,NaN,NaN,2020yCat.1350....0G,RG*,18.059999,0.1026
6,D,K5,18.370001,<NA>,0.0255,14.758,1990A&A...233...21L,NaN,14.833,Could not find a numeric Gaia source_id in 'WE...,NaN,WEL 112,19.945999,NaN,NaN,<NA>,False,A,WEL 112,NaN,NaN,s,NaN,O,NaN,0.0502,15.632,WEL 112,NaN,NaN,90.0,Fnx-21,NaN,NaN,NaN,2020yCat.1350....0G,RG*,17.414,0.0716
7,,,18.59,<NA>,0.1431,15.345,,NaN,15.204,Could not find a numeric Gaia source_id in '[M...,NaN,[MOW91] 25,20.268999,NaN,NaN,<NA>,False,A,[MOW91] 25,NaN,NaN,s,NaN,O,NaN,0.0575,15.924,2MASS J02394711-3431499,NaN,NaN,90.0,Fnx-25,NaN,NaN,NaN,2020yCat.1350....0G,RG*,17.746,0.0782
8,,,19.48,<NA>,NaN,15.937,,NaN,16.462,Could not find a numeric Gaia source_id in '[M...,NaN,[MOV98] 2,NaN,NaN,NaN,<NA>,False,D,[MOV98] 2,NaN,NaN,,NaN,O,NaN,0.2399,16.766001,[MOV98] 2,NaN,18.110001,90.0,Leo-2,NaN,NaN,NaN,2020yCat.1350....0G,*,NaN,0.1984
9,,,19.077999,<NA>,NaN,15.68,,NaN,16.120001,Could not find a numeric Gaia source_id in '[M...,NaN,[MOV98] 5,20.530001,NaN,NaN,<NA>,False,D,[MOV98] 5,NaN,NaN,,NaN,O,NaN,0.2228,16.591999,[MOV98] 5,NaN,17.684999,90.0,Leo-5,NaN,NaN,NaN,2020yCat.1350....0G,RG*,NaN,0.1937


In [14]:
## Sort the Columns
priority_cols = ['Query_ID', 'Found', 'RA_input', 'DEC_input']
coord_cols = [col for col in target_stars_out.columns if any(sub in col for sub in ['RA', 'DEC', 'ra', 'dec'])]
other_cols = [col for col in target_stars_out.columns if col not in priority_cols + coord_cols]
sorted_cols = priority_cols + coord_cols + other_cols
target_stars_out = target_stars_out[sorted_cols]

display(target_stars_out)

,Query_ID,Found,RA_input,DEC_input,ra,parallax_error,RA_input,radial_velocity,pmdec,parallax,DEC_input,dec,pmra,matched_id,sp_qual,sp_bibcode,coo_bibcode,phot_rp_mean_mag,coo_err_angle,J,phot_bp_mean_mag,coo_err_maj,rvz_qual,coo_wavelength,U,V,H,Error,plx_value,B,K,main_id,rvz_nature,R,sp_type,phot_g_mean_mag,I,coo_err_min,source_id,otype
0,[MOP93] 10,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOP93] 10,,,2020yCat.1350....0G,NaN,90,NaN,NaN,0.0598,A,O,NaN,17.906,NaN,Could not find a numeric Gaia source_id in '[M...,NaN,19.152,14.967,[MOP93] 10,s,NaN,,NaN,16.497,0.0613,NaN,RG*
1,[MOP93] 12,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOP93] 12,,,2020yCat.1350....0G,NaN,90,15.596,NaN,0.0658,A,O,NaN,17.907,14.844,Could not find a numeric Gaia source_id in '[M...,0.0487,19.174,14.772,[MOP93] 12,s,NaN,,NaN,16.545,0.0599,NaN,RG*
2,[SMS86] 62,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[SMS86] 62,,,2020yCat.1350....0G,NaN,90,15.151,NaN,0.0565,A,O,NaN,17.684999,14.571,Could not find a numeric Gaia source_id in '[S...,0.0496,19.004999,14.424,[SMS86] 62,sa,NaN,,NaN,16.222,0.0552,NaN,RG*
3,[MOP93] 3,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOP93] 3,,,2020yCat.1350....0G,NaN,90,15.091,NaN,0.057,A,O,NaN,17.684999,14.488,Could not find a numeric Gaia source_id in '[M...,NaN,19.146999,14.275,[MOP93] 3,sa,NaN,,NaN,16.098,0.0518,NaN,RG*
4,[MOP93] 4,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOP93] 4,,,2020yCat.1350....0G,NaN,90,15.231,NaN,0.0583,A,O,NaN,17.625999,14.526,Could not find a numeric Gaia source_id in '[M...,NaN,18.979,14.512,[SMS86] 70,sa,NaN,,NaN,16.181,0.0526,NaN,RR*
5,2MASS J02401001-3429589,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,2MASS J02401001-3429589,,,2020yCat.1350....0G,NaN,90,15.937,NaN,0.0747,A,O,NaN,18.200001,15.198,Could not find a numeric Gaia source_id in '2M...,0.1025,19.76,14.977,2MASS J02401001-3429589,s,18.059999,,NaN,16.75,0.1026,NaN,RG*
6,WEL 112,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,WEL 112,D,1990A&A...233...21L,2020yCat.1350....0G,NaN,90,15.632,NaN,0.0502,A,O,NaN,18.370001,14.833,Could not find a numeric Gaia source_id in 'WE...,0.0255,19.945999,14.758,WEL 112,s,17.414,K5,NaN,NaN,0.0716,NaN,RG*
7,[MOW91] 25,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOW91] 25,,,2020yCat.1350....0G,NaN,90,15.924,NaN,0.0575,A,O,NaN,18.59,15.204,Could not find a numeric Gaia source_id in '[M...,0.1431,20.268999,15.345,2MASS J02394711-3431499,s,17.746,,NaN,NaN,0.0782,NaN,RG*
8,[MOV98] 2,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOV98] 2,,,2020yCat.1350....0G,NaN,90,16.766001,NaN,0.2399,D,O,NaN,19.48,16.462,Could not find a numeric Gaia source_id in '[M...,NaN,NaN,15.937,[MOV98] 2,,NaN,,NaN,18.110001,0.1984,NaN,*
9,[MOV98] 5,False,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,[MOV98] 5,,,2020yCat.1350....0G,NaN,90,16.591999,NaN,0.2228,D,O,NaN,19.077999,16.120001,Could not find a numeric Gaia source_id in '[M...,NaN,20.530001,15.68,[MOV98] 5,,NaN,,NaN,17.684999,0.1937,NaN,RG*


---
## Save the Final Target List as a CSV & FWF
---

In [15]:
## Sort by RA & Dec
# target_stars_out = target_stars_out.sort_values(by=['RA_hms', 'DEC_dms']).reset_index(drop=True)

## Save the target list to a CSV file
target_stars_out.to_csv(os.path.join(script_dir, 'target_stars.csv'), index=False)

## Save the target list to a fixed-width file
target_stars_out = target_stars_out.fillna("---")
target_stars_out.to_string(buf=os.path.join(script_dir, 'target_stars.txt'), index=False)
